# Amino acid substitution discovery

There is a manual database search in the middle of this pipeline: detection
writes a FASTA that has to be searched against the raw files before validation
has anything to read. `Pipeline` exists for that ordering.

In [ ]:
import proteolyzer.aas as aas

params_path = r"params.yaml"
pipeline = aas.Pipeline(params_path)

pipeline.status()   # what has run, and what can run now

## Phase one: preprocess, translate, detect

The six-frame translation is skipped when its frames are already on disk, so
re-running this is cheap.

In [ ]:
pipeline.run_detection()

Now search the raw files against `<output folder>/<sample>_validation.fasta`
and put each search beside the originals as `<sample>_val`.

## Phase two: preprocess, validate, quantify

The preprocessor runs again here to convert those new searches. Phase two
refuses to start before they exist.

In [ ]:
pipeline.run_validation()

## Reading the run back

`Results` finds the artefacts without you knowing the stage-internal file
names. Reading down a column of `summary()` shows where the pipeline stopped.

In [ ]:
results = pipeline.results

print(results.samples)
results.summary()

In [ ]:
results.combined("quantified")   # every sample in one frame

In [ ]:
results.provenance()            # what produced it, and with which parameters

## Driving the stages individually

The orchestrator is only about ordering; each stage still runs on its own.

In [ ]:
import proteolyzer.aas as aas

params_path = r'params.yaml'

preprocessor = aas.Preprocessor.MaxQuant(params_path)
preprocessor.run()

translator = aas.FrameTranslator(params_path)
translator.run()

detector = aas.Detection(params_path)
detector.run()

preprocessor = aas.Preprocessor.MaxQuant(params_path)
preprocessor.run()

validator = aas.Validation(params_path)
validator.run()

quantify = aas.Quantification(params_path)
quantify.run()